# Sketch2Life live AI smoke test

This notebook calls the local Sketch2Life backend with the synthetic FEAT-015 fixture. The backend owns the Lightning credential and provider URL. Never paste a token or raw provider response into a notebook cell or output.

Provider deployment references: https://lightning.ai/docs/platform/inference/inference-overview and https://api.lightning.ai/docs/platform/build/ai-studio/deploy-a-studio

In [ ]:
import hashlib, json, time, urllib.request
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name != 'CAPSTONE' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
FIXTURE = ROOT / 'features/FEAT-015-integration-readiness-review/fixtures/integration-fixture-v1'
BACKEND = 'http://127.0.0.1:8000'
manifest = json.loads((FIXTURE / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['synthetic_data'] is True
hashes = {}
for source in manifest['source_media']:
    digest = hashlib.sha256((FIXTURE / source['path']).read_bytes()).hexdigest()
    assert digest == source['sha256']
    hashes[source['artifact_id']] = digest
print({'fixture_id': manifest['fixture_id'], 'synthetic_data': manifest['synthetic_data'], 'hashes': hashes})

## Start backend first

Use the PowerShell commands in `LIVE_AI_GUIDE.md` to set `SKETCH2LIFE_AI_PROVIDER`, `SKETCH2LIFE_LIGHTNING_AI_BASE_URL`, `SKETCH2LIFE_LIGHTNING_AI_TOKEN_FILE`, and `SKETCH2LIFE_LIVE_FIXTURE_ROOT`, then start Uvicorn. This notebook never needs the provider token.

In [ ]:
payload = {
    'mode': 'live-lightning',
    'fixture_id': manifest['fixture_id'],
    'session_id': 'session-fixture-001',
    'expected_session_version': 1,
}
request = urllib.request.Request(
    f'{BACKEND}/v1/live-understanding',
    data=json.dumps(payload).encode('utf-8'),
    headers={'Content-Type': 'application/json', 'Accept': 'application/json'},
    method='POST',
)
started = time.perf_counter()
with urllib.request.urlopen(request, timeout=30) as response:
    result = json.loads(response.read().decode('utf-8'))
latency_ms = round((time.perf_counter() - started) * 1000, 1)
summary = {
    'status': result['status'],
    'gate_a_required': result['gate_a_required'],
    'proposal_label': result.get('proposal_label'),
    'request_id': result.get('request_id'),
    'asr_contract': result['asr']['contract_name'],
    'vision_contract': result['vision']['contract_name'],
    'provider': result['asr']['provenance']['provider'],
    'model': result['asr']['provenance']['model'],
    'latency_ms': latency_ms,
}
assert result['status'] == 'PROPOSAL'
assert result['gate_a_required'] is True
assert result['asr']['source_audio']['sha256'] == hashes['child-narration-001']
assert result['vision']['source_image']['sha256'] == hashes['child-drawing-001']
print(summary)

## Continue in the UI

Start Metro and the Android emulator. Tap **Try live backend (synthetic fixture)**. The UI must stop at Gate A; confirm it manually, then continue through P1/Gate B, Pixi playback, activity handoff, and feedback. If the backend is unavailable, the UI shows a typed error and fixture mode remains available.